In [ ]:
!nvidia-smi

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
torch.cuda.get_device_name(0)

In [ ]:
import cupy as cp
print(cp.cuda.runtime.getDeviceCount(), "GPU(s) detected")

In [ ]:
import os

print("Exists:", os.path.exists("TMDB_movie_dataset_v112.csv"))

In [ ]:
import cupy as cp
cp.show_config()

In [ ]:
!pip install numba cuda-python -q

In [ ]:
!pip install cupy-cuda12x numba pandas

In [ ]:
!pip install cupy-cuda12x

#128
##16

In [ ]:


import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 128
BLOCK_SIZE_GRID = 128
BLOCKS_PER_GRID = 16

def trim_tail_only(s: str) -> str:
    return s.strip()

def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp

def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0

def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0

def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()
    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)
    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")

def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()
    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)
    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")

def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(year_idx, weights=rating_valid, minlength=YEAR_RANGE).astype(cp.float64)
        d_count_year = cp.bincount(year_idx, minlength=YEAR_RANGE).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(d_genre_ids, weights=d_genre_ratings, minlength=num_genres).astype(cp.float64)
            d_count_genre = cp.bincount(d_genre_ids, minlength=num_genres).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)
        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")

if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 128
blocks_per_grid (auto from N)  = 8192
GPU kernel execution time (block tuning) = 0.155225 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 128
BLOCKS_PER_GRID (number of blocks)  = 16
GPU kernel execution time (grid tuning) = 0.087674 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | 

#128
##20

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 128
BLOCK_SIZE_GRID = 128
BLOCKS_PER_GRID = 20


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 128
blocks_per_grid (auto from N)  = 8192
GPU kernel execution time (block tuning) = 0.171904 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 20 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 128
BLOCKS_PER_GRID (number of blocks)  = 20
GPU kernel execution time (grid tuning) = 0.201179 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | 

#256
##2

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 256
BLOCK_SIZE_GRID = 256
BLOCKS_PER_GRID = 2


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 256
blocks_per_grid (auto from N)  = 4096
GPU kernel execution time (block tuning) = 0.163222 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 256
BLOCKS_PER_GRID (number of blocks)  = 2
GPU kernel execution time (grid tuning) = 0.207494 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#256
##4

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 256
BLOCK_SIZE_GRID = 256
BLOCKS_PER_GRID = 4


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 256
blocks_per_grid (auto from N)  = 4096
GPU kernel execution time (block tuning) = 0.163797 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 256
BLOCKS_PER_GRID (number of blocks)  = 4
GPU kernel execution time (grid tuning) = 0.089572 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#256
##8

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 256
BLOCK_SIZE_GRID = 256
BLOCKS_PER_GRID = 8


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 256
blocks_per_grid (auto from N)  = 4096
GPU kernel execution time (block tuning) = 0.167629 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 256
BLOCKS_PER_GRID (number of blocks)  = 8
GPU kernel execution time (grid tuning) = 0.086329 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#256
##16

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 256
BLOCK_SIZE_GRID = 256
BLOCKS_PER_GRID = 16


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 256
blocks_per_grid (auto from N)  = 4096
GPU kernel execution time (block tuning) = 0.163521 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 256
BLOCKS_PER_GRID (number of blocks)  = 16
GPU kernel execution time (grid tuning) = 0.088579 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | 

#256
##20

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 256
BLOCK_SIZE_GRID = 256
BLOCKS_PER_GRID = 20


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 256
blocks_per_grid (auto from N)  = 4096
GPU kernel execution time (block tuning) = 0.172940 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 20 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 256
BLOCKS_PER_GRID (number of blocks)  = 20
GPU kernel execution time (grid tuning) = 0.087998 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | 

#512
##2

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 512
BLOCK_SIZE_GRID = 512
BLOCKS_PER_GRID = 2


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 512
blocks_per_grid (auto from N)  = 2048
GPU kernel execution time (block tuning) = 0.166067 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 512
BLOCKS_PER_GRID (number of blocks)  = 2
GPU kernel execution time (grid tuning) = 0.087232 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#512
##4

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 512
BLOCK_SIZE_GRID = 512
BLOCKS_PER_GRID = 4


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 512
blocks_per_grid (auto from N)  = 2048
GPU kernel execution time (block tuning) = 0.260191 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE_GRID (threads_per_block) = 512
BLOCKS_PER_GRID (number of blocks)  = 4
GPU kernel execution time (grid tuning) = 0.139649 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endg

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


#512
##8

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 512
BLOCK_SIZE_GRID = 512
BLOCKS_PER_GRID = 8


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 512
blocks_per_grid (auto from N)  = 2048
GPU kernel execution time (block tuning) = 0.163826 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 512
BLOCKS_PER_GRID (number of blocks)  = 8
GPU kernel execution time (grid tuning) = 0.208863 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#512
##16

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 512
BLOCK_SIZE_GRID = 512
BLOCKS_PER_GRID = 16


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 512
blocks_per_grid (auto from N)  = 2048
GPU kernel execution time (block tuning) = 0.165852 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 512
BLOCKS_PER_GRID (number of blocks)  = 16
GPU kernel execution time (grid tuning) = 0.208030 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | 

#512
##20

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 512
BLOCK_SIZE_GRID = 512
BLOCKS_PER_GRID = 20


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 512
blocks_per_grid (auto from N)  = 2048
GPU kernel execution time (block tuning) = 0.164056 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 20 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 512
BLOCKS_PER_GRID (number of blocks)  = 20
GPU kernel execution time (grid tuning) = 0.086576 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | 

#1024
##2

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 1024
BLOCK_SIZE_GRID = 1024
BLOCKS_PER_GRID = 2


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 1024
blocks_per_grid (auto from N)  = 1024
GPU kernel execution time (block tuning) = 0.166137 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 1024
BLOCKS_PER_GRID (number of blocks)  = 2
GPU kernel execution time (grid tuning) = 0.204736 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | 

#1024
##4

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 1024
BLOCK_SIZE_GRID = 1024
BLOCKS_PER_GRID = 4


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 1024
blocks_per_grid (auto from N)  = 1024
GPU kernel execution time (block tuning) = 0.244787 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE_GRID (threads_per_block) = 1024
BLOCKS_PER_GRID (number of blocks)  = 4
GPU kernel execution time (grid tuning) = 0.131998 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: En

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


#1024
#8

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 1024
BLOCK_SIZE_GRID = 1024
BLOCKS_PER_GRID = 8


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 1024
blocks_per_grid (auto from N)  = 1024
GPU kernel execution time (block tuning) = 0.278893 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE_GRID (threads_per_block) = 1024
BLOCKS_PER_GRID (number of blocks)  = 8
GPU kernel execution time (grid tuning) = 0.085644 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: En

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


#1024
##16

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 1024
BLOCK_SIZE_GRID = 1024
BLOCKS_PER_GRID = 16


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 1024
blocks_per_grid (auto from N)  = 1024
GPU kernel execution time (block tuning) = 0.164284 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 1024
BLOCKS_PER_GRID (number of blocks)  = 16
GPU kernel execution time (grid tuning) = 0.086162 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" |

#1024
##20

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 1024
BLOCK_SIZE_GRID = 1024
BLOCKS_PER_GRID = 20


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 1024
blocks_per_grid (auto from N)  = 1024
GPU kernel execution time (block tuning) = 0.167322 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 20 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 1024
BLOCKS_PER_GRID (number of blocks)  = 20
GPU kernel execution time (grid tuning) = 0.501163 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" |